# scVelo, MultiVelo and MultiVelo-AA on Mouse Brain
This notebook applies scVelo, MultiVelo ana MultiVelo-AA to the mouse embryonic brain dataset. The multi-omic methods are applied with varying $w_c$, the parameter which weighs ATAC vs RNA modality during fitting.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import multivelo as mv
import os

import numpy as np
import pandas as pd
import scanpy as sc
import anndata
import scvelo as scv

import matplotlib.pyplot as plt

In [2]:
scv.settings.verbosity = 3
scv.settings.presenter_view = True
scv.set_figure_params('scvelo')
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 200)
np.set_printoptions(suppress=True)

In [3]:
os.makedirs('modeling_results/Mouse_brain/', exist_ok = True)

# Read in data

In [4]:
# RNA modality
adata_rna = sc.read_h5ad('processed_data/adata_rna.h5ad')
# full processed atac for ArchVelo
adata_atac_raw = sc.read_h5ad('processed_data/adata_atac_raw.h5ad')
# aggregated atac for MultiVelo
adata_atac = sc.read_h5ad('processed_data/adata_atac.h5ad')

In [5]:
peak_annotation = pd.read_csv('data/outs/our_peaks/nearest_genes_summits.v2.csv')#pd.read_csv('outs/peak_annotation.tsv', sep = '\t')
peak_annotation_cop = peak_annotation.copy()
peak_annotation_cop.index = [x+':'+str(y)+'-'+str(z) for (x,y,z) in zip(peak_annotation.iloc[:, 0],
                                                             peak_annotation.iloc[:, 1],
                                                             peak_annotation.iloc[:, 2])]
peaks_to_genes = peak_annotation_cop

In [6]:
rna = adata_rna

# Run scvelo

In [9]:
rna_copy = adata_rna.copy()

In [10]:
scv.tl.recover_dynamics(rna_copy, 
                        var_names = rna_copy.var_names,
                        n_jobs = -1)
scv.tl.velocity(rna_copy, mode='dynamical')
scv.tl.velocity_graph(rna_copy)
scv.tl.latent_time(rna_copy)


recovering dynamics (using 128/128 cores)


  0%|          | 0/895 [00:00<?, ?gene/s]

    finished (0:00:53) --> added 
    'fit_pars', fitted parameters for splicing dynamics (adata.var)
computing velocities
    finished (0:00:01) --> added 
    'velocity', velocity vectors for each individual cell (adata.layers)
computing velocity graph (using 1/128 cores)


  0%|          | 0/3365 [00:00<?, ?cells/s]

    finished (0:00:06) --> added 
    'velocity_graph', sparse matrix with cosine correlations (adata.uns)
computing terminal states
    identified 1 region of root cells and 1 region of end points .
    finished (0:00:00) --> added
    'root_cells', root cells of Markov diffusion process (adata.obs)
    'end_points', end points of Markov diffusion process (adata.obs)
computing latent time using root_cells as prior
    finished (0:00:00) --> added 
    'latent_time', shared time (adata.obs)


In [13]:
rna_copy.write('modeling_results/Mouse_brain/scvelo_results.h5ad')

# Smooth full ATAC

In [12]:
# MultiVelo smoothing code applied to full processed ATAC
# need this just for connectivities later

In [15]:
adata_atac_smooth = adata_atac_raw.copy()
# Read in Seurat WNN neighbors.
nn_idx = np.loadtxt("seurat_wnn/nn_idx.txt", delimiter=',')
nn_dist = np.loadtxt("seurat_wnn/nn_dist.txt", delimiter=',')
nn_cells = pd.Index(pd.read_csv("seurat_wnn/nn_cells.txt", header=None)[0])

# Make sure cell names match.
np.all(nn_cells == adata_atac.obs_names)
mv.knn_smooth_chrom(adata_atac_smooth, nn_idx, nn_dist)

In [16]:
nn_idx = np.loadtxt("seurat_wnn/nn_idx.txt", delimiter=',')
nn_dist = np.loadtxt("seurat_wnn/nn_dist.txt", delimiter=',')
nn_cells = pd.Index(pd.read_csv("seurat_wnn/nn_cells.txt", header=None)[0])



In [17]:
adata_atac_smooth.write('processed_data/adata_atac_smooth.h5ad')

# Run MultiVelo with different weights

In [ ]:
# adata_atac.layers['Mc'] = adata_atac.layers['Mc'].todense()

In [ ]:
for wc in np.linspace(0,1,11):
    adata_result = mv.recover_dynamics_chrom(adata_rna, 
                                         adata_atac, 
                                         max_iter=5, 
                                         init_mode="invert", 
                                         verbose=False, 
                                         parallel=True, 
                                         save_plot=False,
                                         rna_only=False,
                                         weight_c = wc,
                                         fit=True,
                                         n_anchors=500, 
                                         n_jobs = -1,
                                         extra_color_key='celltype'
                                        )
    adata_result.write("modeling_results/Mouse_brain/multivelo_result_weight_c_"+str(wc)+".h5ad")

  0%|          | 0/895 [00:00<?, ?it/s]

# Run Multivelo-AA with different weights

In [7]:
## load results of AA on raw ATAC

In [8]:
arch_dir = 'grouping_results/'
rel_dir = arch_dir+'Mouse_brain/archetypes/50_iter/'
XC_train_raw = pd.read_csv(rel_dir+'cell_train_on_peaks_8_comps.csv', index_col = [0])
XC_test_raw = pd.read_csv(rel_dir+'cell_test_on_peaks_8_comps.csv', index_col = [0])
S_raw = pd.read_csv(rel_dir+'peak_on_peaks_8_comps.csv', index_col = [0])

In [9]:
S_raw = S_raw.T
S_raw['gene'] = peak_annotation_cop.loc[S_raw.index,:]['gene']
S_raw.set_index('gene', append = True, inplace = True)
S_raw = S_raw.T

In [10]:
# atac = adata_atac_smooth

In [11]:
rna = adata_rna.copy()

In [12]:
trainc = list(np.ravel(pd.read_csv('processed_data/trainc.csv', index_col = [0]).values))
testc = list(np.ravel(pd.read_csv('processed_data/testc.csv', index_col = [0]).values))

In [13]:
XC_raw = np.zeros((adata_rna.shape[0],8))
XC_raw[trainc,:]=XC_train_raw.values
XC_raw[testc,:]=XC_test_raw.values

In [52]:
# we smooth archetypes over Seurat wnn neighbors
to_smooth = anndata.AnnData(XC_raw.copy(), obs = adata_atac_raw.obs)
mv.knn_smooth_chrom(to_smooth, nn_idx, nn_dist)
XC_from_raw = pd.DataFrame(to_smooth.layers['Mc'], 
             index = adata_rna.obs.index, 
             columns = range(XC_raw.shape[1]))

In [53]:
# from multivelo.dynamical_chrom_func import *

In [54]:
gene_weights_raw = S_raw.T.groupby('gene').mean().T

In [55]:
# create anndata object from XC_from_raw

In [56]:
unimputed_XC_from_raw =  anndata.AnnData(XC_from_raw)
print('imputing')
unimputed_XC_from_raw.layers['spliced'] = unimputed_XC_from_raw.X
unimputed_XC_from_raw.layers['Mc'] = unimputed_XC_from_raw.X

imputing


In [57]:
# Save the result for use later on
unimputed_XC_from_raw.write("processed_data/unimputed_XC_from_raw.h5ad")

In [60]:
prod_from_raw = XC_from_raw @ gene_weights_raw.reset_index(drop = True)
denoised_from_raw = anndata.AnnData(prod_from_raw.values, 
                           obs = adata_atac_raw.obs,
                          var = pd.DataFrame(index = prod_from_raw.columns.values))
denoised_from_raw.layers['Mc'] = denoised_from_raw.X


In [61]:
denoised_from_raw.obsp['connectivities'] = adata_atac_smooth.obsp['connectivities']

In [62]:
# Save the result for use later on
denoised_from_raw.write("processed_data/denoised_from_raw.h5ad")

In [ ]:
rel_vars_cur = denoised_from_raw.var.index.intersection(rna.var.index)
for wc in [0.6]:#np.linspace(0,1,11):
    wc = np.round(wc, 1)
    print(wc)
    full_res_denoised_from_raw  = mv.recover_dynamics_chrom(rna[:,rel_vars_cur], 
                                         denoised_from_raw[:, rel_vars_cur], 
                                         max_iter=5, 
                                         init_mode="invert", 
                                         verbose=False, 
                                         #fit_decoupling=False,
                                         parallel=True, 
                                         save_plot=False,
                                         rna_only=False,
                                         fit=True,
                                         weight_c = wc,
                                         n_anchors=500, 
                                         #extra_color_key='celltype'
                                        )
    full_res_denoised_from_raw.write("modeling_results/Mouse_brain/multivelo_result_denoised_chrom_weight_c_"+str(wc)+".h5ad")

0.6


  0%|          | 0/895 [00:00<?, ?it/s]